In [3]:
import numpy as np

import pyfluids as pf
from pyfluids import FluidsList, Input

VDI external tube bundle median pressure loss:

Definitions:

In [2]:
# Example value for number of rows
n_rows = 11

# Outer tube diameter
d_o    = 26.9/1000

# height of SGH
h      = 1

# Width of SGH
w      = 1

# Full area of SGH
Ac     = h*w

# Inlet mass flow rate
m_dot  = 62.917

# Normal conditions:
T_N    = 15
P_N    = 96600
rho_N  = 1.2888

# Inlet temperature and pressure
P_in    = -4000 + P_N # Pa
T_in    = 230 # degC

# Initial density:
rho     = rho_N *(P_in/P_N)*(T_N/T_in)

# Tube spacing
s_1     = 2*d_o
s_2     = 2*d_o

# Inlet velocity
w_f     = m_dot/(rho*Ac)

# Defining flue gas
air = pf.Fluid(FluidsList.Air)
air.update(Input.temperature(T_in),Input.pressure(P_in))

# Kinematic viscosity
mu      = air.kinematic_viscosity

# Steam inlet pressure and temperature. This should be
T_steam     = 360 + 273.15 # K
P_steam     = 13500000 # Pa

# Calculating correction factors:
air.update(Input.temperature(T_in),Input.pressure(P_in))
eta    = air.dynamic_viscosity

# This should be the mean wall temperature for true
air.update(Input.temperature(T_steam),Input.pressure(P_in))
eta_w  = air.dynamic_viscosity

print(mu)

water_vapour = Fluid(FluidsList.Water).dew_point_at_pressure(101325)

NameError: name 'pf' is not defined

Test data:

In [ ]:
# Inlet temperature and pressure
T_in    = 25 # degC
P_in    = 101325 # Pa

# Defining flue gas
air = pf.Fluid(FluidsList.Air)
air.update(Input.temperature(T_in),Input.pressure(P_in))

# Kinematic viscosity
mu      = air.kinematic_viscosity
rho     = air.density

# Example value for number of rows
n_rows = 2

# Outer tube diameter
d_o    = 38/1000

# Tube spacing
s_1     = 100/1000
s_2     = 92/1000

#Transverse pitch ratio
a   = s_1/d_o

#Longitudinal pitch ratio
b   = s_2/d_o

#Diagonal pitch ratio
c   = ((a/2)**2+b**2)**0.5

#Reynolds number at the narrowest cross-section
Re  = 5000

# Kinematic viscosity


arrangement = "Inline"

# Inlet temperature and pressure
T_water    = 65 # degC

# Calculating correction factors:
air.update(Input.temperature(T_in),Input.pressure(P_in))
eta    = air.dynamic_viscosity

# This should be the mean wall temperature for true
air.update(Input.temperature(T_water),Input.pressure(P_in))
eta_w  = air.dynamic_viscosity

print(mu)

1.5576960431380088e-05


Test model:

In [39]:
#Transverse pitch ratio
a   = s_1/d_o

#Longitudinal pitch ratio
b   = s_2/d_o

#Diagonal pitch ratio
c   = ((a/2)**2+b**2)**0.5

#Mean velocity through narrowest cross-section. w_f is free incoming velocity.
w_e = a/(a-1) * w_f

#Reynolds number at the narrowest cross-section
Re  = (w_e*d_o*rho)/mu
print(mu)

arrangement = "Staggered"

print(1.5*10**6)

4.245263180049891e-05
1500000.0


Code:

In [25]:
# Calculating flow
if Re < 100:
    flow = "laminar"
else:
    flow = "turbulent"

# Correction Factor for Temperature Dependence of Physical Properties in laminar region:
fz_l    = (eta_w/eta)**(0.57/(((((4*a*b)/np.pi) - 1)*Re)**0.25))

# Correction Factor for Temperature Dependence of Physical Properties in turbulent region:
fz_t    = (eta_w/eta)**0.14

# Correction factor for small number of tube rows in laminar region. For n_rows < 10:
if n_rows  < 10:
    fzn_l   = (eta_w/eta)**((0.57*(n_rows/10)**0.25)/(((4*a*b)/np.pi-1)*Re)**0.25)
else:
    fzn_l   = fz_l

if 5 <= n_rows < 10:

    if b >= 1/2 * np.sqrt(2*a+1):
        zeta_o  = 1/(a**2)
        fn_t    = zeta_o*(1/n_rows -1/10)
        print("inline/staggered")

    elif b < 1/2 * np.sqrt(2*a+1):
        zeta_o  = ((2*(c-1))/(a*(a-1)))**2
        fn_t    = zeta_o*(1/n_rows -1/10)
        print("staggered")

elif n_rows >= 10:
    fn_t   = 0

# Calculating pressure drop based on tube arrangement
match arrangement:
    case "Inline":
        print("in line")
        #Number of main resistances for in-line tube arrangement:
        n_MR    = n_rows

        # inline tube arrangement general friction factors for laminar and turbulent flow:
        F_f     = 1-np.exp(-(Re+1000)/2000)
        f_a_lf  = (280*np.pi*((b**0.5-0.6)**2+0.75))/((4*a*b-np.pi)*a**1.6)
        f_a_tf  = (0.22+1.2*(((1-0.94/b)**0.6)/(a-0.85)**1.3))*10**(0.47*((b/a)-1.5))+(0.03*(a-1)*(b-1))
        
        zeta_lam    = f_a_lf/Re
        zeta_turb   = f_a_tf/(Re**(0.1*(b/a)))

        match flow:
            case "laminar":
                zeta        = zeta_lam*fzn_l

            case "turbulent":
                zeta        = zeta_lam*fzn_l + (zeta_turb*fz_t+fn_t)*F_f

        Delta_P = zeta * n_MR*((rho*w_e**2)/(2))
        print(Delta_P)
        
    case "Staggered":
        print("staggered")
        #Number of main resistances:
        if b >= 0.5*np.sqrt(2*a+1):
            n_MR = n_rows

        elif b < 0.5*np.sqrt(2*a+1):
            n_MR = n_rows -1

        #in line tube arrangement:
        F_v    = 1-np.exp(-(Re+200)/1000)

        if b >= 0.5*np.sqrt(2*a+1):
            f_a_lv = (280*np.pi*((b**0.5-0.6)**2+0.75))/((4*a*b-np.pi)*a**1.6)

        elif b < 0.5*np.sqrt(2*a+1):
            f_a_lv = (280*np.pi*((b**0.5-0.6)**2+0.75))/((4*a*b-np.pi)*c**1.6)

        f_a_tv  = 2.5+(1.2/((a-0.85)**1.08))+0.4*(b/a-1)**3-0.01*(a/b-1)**3

        zeta_lam    = f_a_lv/Re
        zeta_turb   = f_a_tv/(Re**(0.1*(b/a)))

        match flow:
            case "laminar":
                zeta        = zeta_lam*fzn_l

            case "turbulent":
                zeta        = zeta_lam*fzn_l + (zeta_turb*fz_t+fn_t)*F_v    

        Delta_P = zeta * n_MR*((rho*w_e**2)/(2))
        print(Delta_P)

        print(fn_t)

in line
40.26046550498652


In line arrangement:

In [311]:
#Number of main resistances:
n_MR    = n_rows

#in line tube arrangement:
F_f     = 1-np.exp((-Re+1000)/2000)
f_a_lf  = (280*np.pi*((b**0.5-0.6)**2+0.75))/((4*a*b-np.pi)*a**1.6)
f_a_tf  = (0.22+1.2*(((1-0.94/b)**0.6)/(a-0.85)**1.3))*10**(0.47*((b/a)-1.5))+(0.03*(a-1)*(b-1))

zeta_lam    = f_a_lf/Re
zeta_turb   = f_a_tf/(Re**(0.1*(b/a)))

zeta        = zeta_lam + zeta_turb*F_f

Delta_P = zeta * n_MR*((rho*w_e**2)/(2))

print(Delta_P)

11.434681499802508


Staggered arrangment:

In [312]:
#Number of main resistances:
if b >= 0.5*np.sqrt(2*a+1):
    n_MR = n_rows

elif b < 0.5*np.sqrt(2*a+1):
    n_MR = n_rows -1


#in line tube arrangement:
F_v    = 1-np.exp((-Re+200)/1000)

if b >= 0.5*np.sqrt(2*a+1):
    f_a_lv = (280*np.pi*((b**0.5-0.6)**2+0.75))/((4*a*b-np.pi)*a**1.6)

elif b < 0.5*np.sqrt(2*a+1):
    f_a_lv = (280*np.pi*((b**0.5-0.6)**2+0.75))/((4*a*b-np.pi)*c**1.6)

f_a_tv  = 2.5+(1.2/((a-0.85)**1.08))+0.4*(b/a-1)**3-0.01*(a/b-1)**3

zeta_lam    = f_a_lv/Re
zeta_turb   = f_a_tv/(Re**(0.1*(b/a)))

zeta        = zeta_lam + zeta_turb*F_v

Delta_P = zeta * n_MR*((rho*w_e**2)/(2))

print(Delta_P)

86.93903549384675
